# Korean spacing correction: LoRA fine-tuning on a free GPU

Runs the whole experiment from this repository on **one free 16 GB GPU**.

**Kaggle:** Settings → Accelerator → *GPU T4 x2* (the notebook uses one), Internet → *On*.
**Colab:** Runtime → Change runtime type → *T4 GPU*.

Rough time on a T4: zero-shot evaluation about 15 min, training about 45–60 min, fine-tuned evaluation about 15 min.
If the session disconnects, run the cells again; training resumes from the last checkpoint (`--resume`).

Optional: add a Hugging Face write token as a secret named `HF_TOKEN` to upload the adapter at the end.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use a single GPU even on "T4 x2"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
os.chdir("/kaggle/working" if os.path.isdir("/kaggle/working") else "/content")
!git clone -q https://github.com/keshri-prasanjeet/korean-text-restoration.git || (cd korean-text-restoration && git pull -q)
%cd korean-text-restoration
!pip install -q -e ".[train]"
!python -m pytest -q

## 1. Zero-shot baseline (base model, no fine-tuning)

In [ ]:
!python scripts/evaluate_llm.py --base-model Qwen/Qwen2.5-1.5B-Instruct --name qwen2.5-1.5b-zeroshot

## 2. LoRA fine-tuning (QLoRA, 4-bit base)

In [ ]:
!python scripts/train_lora.py --base-model Qwen/Qwen2.5-1.5B-Instruct --train-size 20000 \
    --output-dir outputs/qwen2.5-1.5b-spacing --load-in-4bit --resume

## 3. Evaluate the fine-tuned model

In [ ]:
!python scripts/evaluate_llm.py --base-model Qwen/Qwen2.5-1.5B-Instruct \
    --adapter outputs/qwen2.5-1.5b-spacing/adapter --name qwen2.5-1.5b-lora

In [ ]:
!python scripts/make_table.py

## 4. Save the results

Download `results.zip` (Kaggle: *Output* panel; Colab: the file browser), unzip it in the repository root, and commit `results/`.
The adapter is in `outputs/` and can be uploaded with the next cell.

In [ ]:
!cp outputs/qwen2.5-1.5b-spacing/training_log.json results/qwen2.5-1.5b-lora-training_log.json
!zip -qr results.zip results
!zip -qr adapter.zip outputs/qwen2.5-1.5b-spacing/adapter
!ls -lh results.zip adapter.zip

In [ ]:
# Optional: upload the LoRA adapter to the Hugging Face Hub.
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass

if token:
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    repo_id = f"{api.whoami()['name']}/qwen2.5-1.5b-ko-spacing-lora"
    api.create_repo(repo_id, exist_ok=True)
    api.upload_folder(folder_path="outputs/qwen2.5-1.5b-spacing/adapter", repo_id=repo_id)
    print("uploaded to https://huggingface.co/" + repo_id)
else:
    print("No HF_TOKEN secret found; skipping upload.")